# 🤖 Notebook 3 — Model Training

**Course:** PA2595 Machine Learning Engineering

---

## What is model training?

**Training** a machine learning model means letting an algorithm learn patterns from labelled data.

We give the model:
- **Input (X):** feature values for each student (study time, absences, grades, etc.)
- **Output (y):** the correct label for each student (1 = Pass, 0 = Fail)

The model adjusts its internal parameters until it can map inputs to outputs as accurately as possible.

---

## Which models do we compare?

| Model | How it works |
|---|---|
| **Decision Tree** | Asks a series of yes/no questions about the features — like a flowchart |
| **Random Forest** | Builds many decision trees and combines their votes — more accurate and robust |
| **Logistic Regression** | Calculates a weighted sum of features and maps it to a probability (0–1) |

> ⚠️ Run notebook `02_preprocessing.ipynb` first to generate the files in `data/processed/`.

## Step 1 — Import Libraries

In [ ]:
import sys
import os
sys.path.insert(0, os.path.join(os.getcwd(), ".."))

import pandas as pd
import joblib
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

%matplotlib inline
print("Libraries loaded.")

## Step 2 — Load Preprocessed Data

In [ ]:
X_train = pd.read_csv("../data/processed/X_train.csv")
y_train = pd.read_csv("../data/processed/y_train.csv").squeeze()

print(f"Training samples : {len(X_train)}")
print(f"Features         : {X_train.shape[1]}")
print(f"Pass in training : {y_train.sum()} ({y_train.mean():.1%})")

## Step 3 — Model 1: Decision Tree

### How does it work?

A Decision Tree splits the data by asking questions like:
> "Is G2 >= 10? If yes → likely Pass. If no → ask another question."

It keeps splitting until it reaches a **leaf node** — a final prediction.

### Key hyperparameters:
- **`max_depth`**: how many levels of questions the tree can ask.  
  A very deep tree memorises the training data (overfitting). We limit it to 5.
- **`random_state`**: fixes the random seed so results are reproducible.

In [ ]:
dt_model = DecisionTreeClassifier(max_depth=5, random_state=42)
dt_model.fit(X_train, y_train)

train_accuracy = dt_model.score(X_train, y_train)
print(f"Decision Tree - Training accuracy: {train_accuracy:.4f}")

### Visualise the Decision Tree

This is one of the great advantages of Decision Trees — you can actually **see** the decisions the model makes.

Each node shows:
- The **question** being asked (feature and threshold)
- **gini**: a measure of impurity — 0 means all samples in this node have the same label
- **samples**: how many training samples reach this node
- **value**: `[Fail count, Pass count]`

In [ ]:
fig, ax = plt.subplots(figsize=(20, 8))
plot_tree(
    dt_model,
    feature_names=X_train.columns.tolist(),
    class_names=["Fail", "Pass"],
    filled=True,
    rounded=True,
    max_depth=3,   # show only first 3 levels to keep it readable
    ax=ax,
    fontsize=9
)
ax.set_title("Decision Tree (first 3 levels shown)", fontsize=14)
plt.tight_layout()
plt.show()

## Step 4 — Model 2: Random Forest

### How does it work?

Random Forest builds **many decision trees** (here: 100), each trained on:
- A random subset of the training samples (**bootstrapping**)
- A random subset of the features at each split

The final prediction is the **majority vote** across all trees.  
This reduces the tendency to overfit that a single tree has.

### Key hyperparameters:
- **`n_estimators`**: number of trees. More trees = more stable, but slower.
- **`max_depth`**: maximum depth of each individual tree.
- **`n_jobs=-1`**: use all available CPU cores to train in parallel.

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)

train_accuracy = rf_model.score(X_train, y_train)
print(f"Random Forest - Training accuracy: {train_accuracy:.4f}")

### Feature Importance

Random Forest can tell us which features were **most useful** for making predictions.  
Higher importance score = the feature contributed more to the predictions.

In [ ]:
importances = pd.Series(
    rf_model.feature_importances_,
    index=X_train.columns
).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 8))
importances.plot(kind="barh", ax=ax, color="#3498db", edgecolor="black")
ax.set_title("Random Forest - Feature Importances", fontsize=14)
ax.set_xlabel("Importance Score")
plt.tight_layout()
plt.show()

print("\nTop 5 most important features:")
print(importances.sort_values(ascending=False).head(5).to_string())

## Step 5 — Model 3: Logistic Regression

### How does it work?

Despite the name, Logistic Regression is a **classification** algorithm, not regression.

It computes a weighted sum of all input features, then passes it through the **sigmoid function** to get a probability between 0 and 1. If probability >= 0.5 → Pass, else → Fail.

### Key hyperparameters:
- **`max_iter`**: maximum number of optimisation steps. We set it high (1000) to ensure convergence.
- **`solver`**: the algorithm used for optimisation. `lbfgs` is reliable for small datasets.

In [ ]:
lr_model = LogisticRegression(max_iter=1000, random_state=42, solver="lbfgs")
lr_model.fit(X_train, y_train)

train_accuracy = lr_model.score(X_train, y_train)
print(f"Logistic Regression - Training accuracy: {train_accuracy:.4f}")

### Logistic Regression Coefficients

Each feature has a **coefficient (weight)** that shows how much it pushes the prediction:
- **Positive coefficient** → pushes toward Pass
- **Negative coefficient** → pushes toward Fail

In [ ]:
coef = pd.Series(
    lr_model.coef_[0],
    index=X_train.columns
).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 8))
colors = ["#e74c3c" if v < 0 else "#2ecc71" for v in coef]
coef.plot(kind="barh", ax=ax, color=colors, edgecolor="black")
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title("Logistic Regression - Feature Coefficients", fontsize=14)
ax.set_xlabel("Coefficient value  (positive = pushes toward Pass)")
plt.tight_layout()
plt.show()

## Step 6 — Training Accuracy Comparison

In [ ]:
summary = pd.DataFrame({
    "Model": ["Decision Tree", "Random Forest", "Logistic Regression"],
    "Train Accuracy": [
        dt_model.score(X_train, y_train),
        rf_model.score(X_train, y_train),
        lr_model.score(X_train, y_train),
    ]
}).set_index("Model")

print(summary.round(4).to_string())
print("\nNote: high training accuracy can indicate overfitting - evaluate on the test set in notebook 04.")

## Step 7 — Save the Trained Models

We save each model to the `models/` folder using **joblib**.  
This means we don't have to retrain every time — we can simply load the saved file.

In [ ]:
MODELS_DIR = "../models"
os.makedirs(MODELS_DIR, exist_ok=True)

joblib.dump(dt_model, f"{MODELS_DIR}/decision_tree.pkl")
joblib.dump(rf_model, f"{MODELS_DIR}/random_forest.pkl")
joblib.dump(lr_model, f"{MODELS_DIR}/logistic_regression.pkl")
joblib.dump(list(X_train.columns), f"{MODELS_DIR}/feature_columns.pkl")

print("Saved models:")
for f in os.listdir(MODELS_DIR):
    print(f"  {f}")

## ✅ Summary

| Model | Key idea | Interpretable? |
|---|---|---|
| Decision Tree | Step-by-step yes/no questions | ✅ Very easy to explain |
| Random Forest | Combines 100 trees by majority vote | ⚠️ Hard to explain individual decisions |
| Logistic Regression | Linear weighted sum + sigmoid | ✅ Coefficients are interpretable |

> 📌 **Next step:** Open notebook `04_evaluation.ipynb` to compare the models on the **test set** and pick the best one.